# Avalanche buildup time and current slope vs overvoltage
For every device configuration (folder prefix) and bias, the seeds are filtered with a
Savitzky-Golay filter and flagged as *avalanche* when the filtered current stays above
`I_avalanche`. For the biases where most seeds develop the avalanche, it computes

- the **avalanche buildup time (ABT)**: first time the filtered current reaches `I_threshold`;
- the **relative current slope**: slope of $\log_{10}$ of the filtered current over its last points.

Figures are saved in `SAVE_DIR` as `.png`, `.pdf` and editable `.mplfig`
(reopen with `python open_fig.py <file>.mplfig` or `mcplot.load_figure`).

In [ ]:
# Interactive figures (zoom/pan) need ipympl; otherwise fall back to static figures
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
except ImportError:
    get_ipython().run_line_magic("matplotlib", "inline")
import os

import matplotlib.pyplot as plt
import numpy as np
import re

import matplotlib.lines as mlines
import pandas as pd
from scipy.signal import savgol_filter

from mcplot import extract_vdc_from_folder, find_seed_folders, read_current_voltage, seed_number
from mcplot import save_figure, set_style

set_style(24)

SAVE_FIGURES = True  # save every figure as .png, .pdf and editable .mplfig
SAVE_DIR = "output"


def save(fig, name):
    """Save fig as SAVE_DIR/name.{png,pdf,mplfig} (reopen the .mplfig with open_fig.py)."""
    if SAVE_FIGURES:
        save_figure(fig, os.path.join(SAVE_DIR, name), formats=("png", "pdf"), dpi=300, bbox_inches="tight")

# Simulation folders
Searched as `<base_path>/<prefix>-<V>V-s<seed>`; the label is the number in the prefix, e.g. `deep_implant_1.0` → `1.0 µm`.

In [ ]:
base_path = "/home/angelo/server/csmhome2/amudano/MonteCarlo/DEMC/SPAD_FBK_v2/quasi_1D"
voltages = np.linspace(-30, -40, 11)
subfolder_prefixes = [
    "FF/deep_implant_1.0/breakdown",
    "FF/deep_implant_1.1/breakdown",
    "FF/deep_implant_1.2/breakdown",
    "FF/deep_implant_1.3/breakdown",
    # "FF/deep_implant_1.4/breakdown",
    # "FF/deep_implant_1.5/breakdown",
    # "FF/deep_implant_1.6/breakdown",
    # "FF/deep_implant_1.7/breakdown",
]
contact = "ncontact"


def thickness_label(prefix):
    """"FF/deep_implant_1.0/breakdown" -> "1.0 µm"."""
    raw = prefix.split("/")[-2] if "/" in prefix else prefix
    match = re.search(r"_(\d+\.?\d*)", raw)
    return f"{match.group(1)} µm" if match else raw


folders_by_prefix = find_seed_folders(base_path, subfolder_prefixes, voltages, label_fn=thickness_label)

# Data post processing

In [ ]:
I_avalanche = 1e-5  # avalanche: filtered current above I_avalanche for n_consecutive samples
n_consecutive = 10
I_threshold = 1e-6  # ABT: first time the filtered current reaches I_threshold
n_fit_last = 2000  # last samples of the filtered current used for the slope fit

# Savitzky-Golay filter
sv_time_window_ps = 0.5  # target window length (ps)
time_step_fs = 8.0  # sampling step of the current (fs)
polyorder = 3
target_window_pts = int(sv_time_window_ps * 1000 / time_step_fs)
if target_window_pts % 2 == 0:
    target_window_pts += 1


def sv_window(n_points):
    """Target window, reduced to n_points // 5 (odd, > polyorder + 1) for short transients."""
    if target_window_pts <= n_points // 5:
        return target_window_pts
    window = max(n_points // 5, polyorder + 2)
    return window + 1 if window % 2 == 0 else window


# data_dict[prefix][vdc][seed] = {"flag", "current", "current_filt", "time"}
data_dict = {}
for prefix, folders in folders_by_prefix.items():
    data_dict[prefix] = {}
    print(f"\nScanning prefix: {prefix}")
    for folder in folders:
        seeds = data_dict[prefix].setdefault(extract_vdc_from_folder(folder), {})
        current_file = f"{base_path}/{folder}/contact_current.csv"
        try:
            current, _, time_s, _ = read_current_voltage(current_file, contact)
        except Exception:
            print(f"  Error reading {current_file}")
            continue
        curr = np.abs(current.to_numpy())
        curr_filt = savgol_filter(curr, sv_window(len(curr)), polyorder)
        n_above = np.convolve((curr_filt > I_avalanche).astype(int), np.ones(n_consecutive), mode="valid")
        seeds[seed_number(folder)] = {
            "flag": bool(np.any(n_above == n_consecutive)),
            "current": curr,
            "current_filt": curr_filt,
            "time": time_s.to_numpy(),
        }

## Filter calibration
Raw and filtered current of a few avalanche seeds at the highest |V| (fastest avalanche) of the first prefix.

In [ ]:
plt.close("all")
n_test_seeds = 3
test_prefix = next(iter(data_dict))
valid_vdcs = [v for v, seeds in data_dict[test_prefix].items() if any(s["flag"] for s in seeds.values())]
test_vdc = max(valid_vdcs, key=abs)
test_seeds = [(seed, s) for seed, s in data_dict[test_prefix][test_vdc].items() if s["flag"]][:n_test_seeds]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["r", "b", "g", "m", "c"]
for i, (seed, s) in enumerate(test_seeds):
    c = colors[i % len(colors)]
    t_ps = s["time"] * 1e12
    ax.semilogy(t_ps, s["current"], color=c, alpha=0.3, linewidth=1,
                label=f"Seed {seed} (Raw)" if i == 0 else "_nolegend_")
    ax.semilogy(t_ps, s["current_filt"], color=c, linewidth=2.5, label=f"Seed {seed} (Filtered)")
ax.axhline(I_threshold, color="k", linestyle=":", label="I_threshold")
ax.set_xlabel("Time (ps)")
ax.set_ylabel("Current (A)")
ax.set_xlim(left=0)
ax.grid(True, which="both", alpha=0.5)
ax.legend(fontsize=12)
fig.tight_layout()
save(fig, "test_filter")
plt.show()

## Buildup time and slope
A bias is used when more than half of its seeds develop the avalanche; overvoltage 1 V is the lowest such |V|.

In [ ]:
results_by_prefix = {}
for prefix, vdc_dict in data_dict.items():
    valid_vdcs = sorted((v for v, seeds in vdc_dict.items()
                         if sum(s["flag"] for s in seeds.values()) >= len(seeds) // 2 + 1), key=abs)
    if not valid_vdcs:
        continue
    min_abs_vdc = abs(valid_vdcs[0])

    res = {k: [] for k in ("vov", "avg_time", "std_time", "avg_slope", "std_slope")}
    avg_t_by_vdc = {}
    for vdc in sorted(vdc_dict, key=abs):
        times, slopes = [], []
        for s in vdc_dict[vdc].values():
            if not s["flag"]:
                continue
            above = np.where(s["current_filt"] >= I_threshold)[0]
            if len(above) > 0:
                times.append(s["time"][above[0]] * 1e12)  # ps
            n_last = min(n_fit_last, len(s["current_filt"]))
            if n_last > 1:
                slope, _ = np.polyfit(s["time"][-n_last:], np.log10(s["current_filt"][-n_last:]), 1)
                slopes.append(slope)

        avg_t_by_vdc[vdc] = None
        if times and vdc in valid_vdcs:
            avg_t_by_vdc[vdc] = np.mean(times)
            res["vov"].append(abs(vdc) - min_abs_vdc + 1)
            res["avg_time"].append(np.mean(times))
            res["std_time"].append(np.std(times))
            res["avg_slope"].append(np.mean(slopes) if slopes else 0)
            res["std_slope"].append(np.std(slopes) if slopes else 0)

    results_by_prefix[prefix] = {k: np.array(v) for k, v in res.items()}
    results_by_prefix[prefix]["avg_t_by_vdc"] = avg_t_by_vdc

print("\nAvalanche buildup time computed.")

# Debug plots
Current of every seed (grey: no avalanche) with `I_threshold` and the average ABT.

In [ ]:
def plot_seed_currents(ax, seeds, avg_t, avg_label="avg_t"):
    for s in seeds.values():
        if s["flag"]:
            ax.semilogy(s["time"] * 1e12, s["current"], alpha=0.6, linewidth=1.5)
        else:
            ax.semilogy(s["time"] * 1e12, s["current"], color="lightgray", alpha=0.5, linewidth=1)
    ax.axhline(I_threshold, color="r", linestyle="--", label=f"I_th={I_threshold:.0e}")
    if avg_t is not None:
        ax.axvline(avg_t, color="b", linestyle="-.", label=f"{avg_label}={avg_t:.1f} ps")
    ax.set_xlabel("Time (ps)")
    ax.set_ylabel("Current (A)")
    ax.grid(True, which="both", linestyle=":", alpha=0.6)
    ax.legend(fontsize=24, loc="best")

In [ ]:
plt.close("all")
n_cols = 2
for prefix, vdc_dict in data_dict.items():
    if prefix not in results_by_prefix:
        continue
    vdcs = sorted(vdc_dict, key=abs)
    n_rows = int(np.ceil(len(vdcs) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 5 * n_rows), squeeze=False)
    fig.suptitle(f"Debug: {prefix}", fontsize=16, fontweight="bold")
    axes = axes.flatten()
    for ax, vdc in zip(axes, vdcs):
        plot_seed_currents(ax, vdc_dict[vdc], results_by_prefix[prefix]["avg_t_by_vdc"][vdc])
        ax.set_title(f"V = {vdc} V")
    for ax in axes[len(vdcs):]:
        ax.set_visible(False)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    save(fig, f"debug_plot_{prefix}")
plt.show()

## Single bias

In [ ]:
plt.close("all")
target_vdc = -34.0
for prefix, vdc_dict in data_dict.items():
    if prefix not in results_by_prefix:
        continue
    if target_vdc not in vdc_dict:
        print(f"{prefix}: no simulations at {target_vdc} V")
        continue
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_seed_currents(ax, vdc_dict[target_vdc], results_by_prefix[prefix]["avg_t_by_vdc"][target_vdc],
                       avg_label="ABT")
    ax.set_title(f"V = {target_vdc} V")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    save(fig, f"debug_plot_{prefix}_V{target_vdc:g}")
plt.show()

## Single curve
Current of one seed at one bias. With `seed = None` the first seed that develops the avalanche is used.

In [ ]:
plt.close("all")
curve_prefix = "1.0 µm"  # one of data_dict.keys()
curve_vdc = -34.0
seed = None  # seed number, or None for the first avalanche seed
filtered = False  # True: Savitzky-Golay filtered current

seeds = data_dict[curve_prefix][curve_vdc]
if seed is None:
    seed = next(n for n, s in seeds.items() if s["flag"])
s = seeds[seed]
current = s["current_filt"] if filtered else s["current"]

# ABT of this seed: first time the filtered current reaches I_threshold (as in the analysis)
above = np.where(s["current_filt"] >= I_threshold)[0]
abt = s["time"][above[0]] * 1e12 if len(above) > 0 else None

fig, ax = plt.subplots(figsize=(8, 6))
ax.semilogy(s["time"] * 1e12, current, linewidth=2, label=f"{curve_prefix}, V = {curve_vdc:g} V, seed {seed}")
ax.axhline(I_threshold, color="r", linestyle="--", label=f"I_th={I_threshold:.0e} A")
if abt is not None:
    ax.axvline(abt, color="b", linestyle="-.", label=f"ABT={abt:.1f} ps")
else:
    print(f"Seed {seed} never reaches I_threshold: no ABT")
ax.set_xlabel("Time (ps)")
ax.set_ylabel("Current (A)")
ax.grid(True, which="both", linestyle=":", alpha=0.6)
ax.legend(fontsize=16)
fig.tight_layout()
save(fig, f"current_{curve_prefix}_V{curve_vdc:g}_s{seed}")
plt.show()

# Results vs overvoltage
One marker/line style per prefix; ABT in blue, relative current slope in orange.

In [ ]:
prefix_colors = plt.cm.tab10.colors
prefix_markers = ["o", "s", "^", "v", "P", "*", "X", "D", "h"]
prefix_lines = ["--", "-.", ":", "-"]


def plot_vs_overvoltage(ax, key, scale, color):
    for idx, (label, data) in enumerate(results_by_prefix.items()):
        fmt = prefix_markers[idx % len(prefix_markers)] + prefix_lines[idx % len(prefix_lines)]
        ax.plot(data["vov"], data[key] * scale, fmt, markersize=10, linewidth=3, color=color, label=label)


def prefix_legend(ax, fontsize=28):
    """Legend with one black marker/line style per prefix."""
    handles = [mlines.Line2D([], [], color="black", markersize=10, label=label,
                             marker=prefix_markers[i % len(prefix_markers)],
                             linestyle=prefix_lines[i % len(prefix_lines)])
               for i, label in enumerate(results_by_prefix)]
    ax.legend(handles=handles, ncol=max(len(handles), 1), loc="upper center", fontsize=fontsize)

## Buildup time and current slope

In [ ]:
plt.close("all")
fig, ax1 = plt.subplots(figsize=(16, 10))
ax2 = ax1.twinx()
plot_vs_overvoltage(ax1, "avg_time", 1, prefix_colors[0])
plot_vs_overvoltage(ax2, "avg_slope", 1e-12, prefix_colors[1])
ax1.set_xlabel("Overvoltage, V", fontsize=40)
ax1.grid(True, linestyle="--", alpha=0.7)
prefix_legend(ax1)
for ax, color, ylabel, side, other in [(ax1, prefix_colors[0], "Avalanche buildup time, ps", "left", "right"),
                                       (ax2, prefix_colors[1], "Relative current slope, 1/ps", "right", "left")]:
    ax.spines[side].set_color(color)
    ax.spines[other].set_visible(False)
    ax.tick_params(axis="y", colors=color)
    ax.set_ylabel(ylabel, fontsize=40, color=color)
fig.tight_layout()
save(fig, "avalanche_buildup/buildup_time_vs_overvoltage")
plt.show()

In [ ]:
# Save the results of each prefix to a csv file
csv_dir = os.path.join(SAVE_DIR, "avalanche_buildup")
os.makedirs(csv_dir, exist_ok=True)
for prefix, data in results_by_prefix.items():
    path = os.path.join(csv_dir, f"{prefix}_buildup.csv")
    pd.DataFrame({
        "Overvoltage_V": data["vov"],
        "Avg_Time_to_threshold_ps": data["avg_time"],
        "Std_Time_to_threshold_ps": data["std_time"],
        "Avg_Rel_current_slope": data["avg_slope"],
        "Std_Rel_current_slope": data["std_slope"],
    }).to_csv(path, index=False)
    print(f"Saved: {path}")

## Buildup time

In [ ]:
plt.close("all")
fig, ax = plt.subplots(figsize=(16, 10))
plot_vs_overvoltage(ax, "avg_time", 1, prefix_colors[0])
ax.set_xlabel("Overvoltage, V", fontsize=28)
ax.set_ylabel("Avalanche buildup time, ps", fontsize=28)
ax.grid(True, linestyle="--", alpha=0.7)
prefix_legend(ax)
fig.tight_layout()
save(fig, "avalanche_buildup/buildup_time")
plt.show()

## Relative current slope

In [ ]:
plt.close("all")
fig, ax = plt.subplots(figsize=(16, 10))
plot_vs_overvoltage(ax, "avg_slope", 1e-12, prefix_colors[1])
ax.set_xlabel("Overvoltage, V", fontsize=28)
ax.set_ylabel("Relative current slope, dec/ps", fontsize=28)
ax.grid(True, linestyle="--", alpha=0.7)
prefix_legend(ax)
fig.tight_layout()
save(fig, "avalanche_buildup/relative_current_slope")
plt.show()